# Notebook 1 — Descarga de datos iniciales
## Líneas espectrales y niveles de energía — NIST ASD
## Proyecto: Óptica y Fotónica — Primer Parcial

**Estudiante:** Perez Criollo Andres David
**Fuente oficial (líneas):** https://physics.nist.gov/PhysRefData/ASD/lines_form.html
**Fuente oficial (niveles):** https://physics.nist.gov/PhysRefData/ASD/levels_form.html
**Endpoints:** `lines1.pl` y `energy1.pl`
**Fecha de descarga:** (completar al ejecutar)

Este notebook reúne la obtención de las **dos** fuentes del NIST (antes en dos archivos). No mezcla los CSV: cada parte escribe los mismos archivos que antes.

| Parte | Qué descarga | Archivos (sin modificar después) |
|---|---|---|
| A | Líneas espectrales (H–Ne, I y II) | `datos_originales/nist_lines_H_Ne_original.csv` |
| B | Niveles de energía (18 espectros) | `datos_originales/nist_levels_todos_original.csv` y `datos_originales/niveles/` |

Los notebooks siguientes (exploración, limpieza, ETL, consultas) leen esos CSV. **No hace falta re-ejecutar esta descarga** si los archivos ya existen: re-pedirlos al NIST podría reemplazar la evidencia que ya usa el resto del proyecto.

Ejecuta las celdas **de arriba hacia abajo**.


---
## Celda 1 — Instalación de librerías

In [ ]:
# Instalacion de librerias necesarias
# pandas  -> manejo y analisis de datos tabulares
# requests -> hacer peticiones HTTP a la API del NIST

!pip install pandas requests

---
## Celda 2 — Importación de librerías

In [ ]:
import requests          # Para hacer las peticiones HTTP a la API del NIST
import pandas as pd      # Para leer los CSV y manipular los datos
from io import StringIO  # Para convertir texto de respuesta en objeto legible por pandas
import os               # Para crear carpetas en el sistema de archivos
import time             # Para pausas entre peticiones (buena practica con APIs)
from datetime import datetime  # Para registrar la fecha de descarga

print("Librerias importadas correctamente.")
print(f"Fecha y hora de ejecucion: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## Crear carpetas de datos originales

Se crean `datos_originales/` y `datos_originales/niveles/`. No se modifican archivos ya existentes hasta que cada parte los escriba al final, igual que en las descargas originales.


In [ ]:
# Carpeta principal de datos originales
os.makedirs("datos_originales", exist_ok=True)

# Subcarpeta especifica para los niveles de energia (un archivo por espectro)
os.makedirs("datos_originales/niveles", exist_ok=True)

print("Carpetas creadas:")
print("  datos_originales/")
print("  datos_originales/niveles/")

---
# Parte A — Líneas espectrales (`lines1.pl`)

Una petición con los 18 espectros juntos. Cada fila es una transición electrónica.


---
## Celda 4 — Definición de la URL y los parámetros de consulta

Cada parámetro corresponde exactamente a una opción del formulario web en:  
https://physics.nist.gov/PhysRefData/ASD/lines_form.html

In [ ]:
# URL del endpoint de la API del NIST ASD para lineas espectrales
URL = "http://physics.nist.gov/cgi-bin/ASD/lines1.pl"

# Elementos a consultar: estado neutro (I) y primer ionizado (II)
# de los primeros 10 elementos de la tabla periodica
# El punto y coma (;) es el separador entre espectros en la API del NIST
ESPECTRA = "H I;He I;Li I;Li II;Be I;Be II;B I;B II;C I;C II;N I;N II;O I;O II;F I;F II;Ne I;Ne II"

parametros = {

    # --- CAMPO PRINCIPAL ---
    "spectra": ESPECTRA,
    # Que elementos y estados de ionizacion consultar
    # Equivale al campo 'Spectrum' en el formulario web

    # --- RANGO DE LONGITUD DE ONDA ---
    "low_w": "",
    # Longitud de onda minima — se deja vacia para obtener TODAS
    # Si se pusiera un valor (ej. 200), solo traeria lineas >= 200 nm

    "upp_w": "",
    # Longitud de onda maxima — se deja vacia para obtener TODAS
    # Equivale al campo 'Upper' en el formulario web

    # --- UNIDADES DE LONGITUD DE ONDA ---
    "unit": 1,
    # 0 = Angstroms (Å)
    # 1 = Nanometros (nm)  <-- SELECCIONADO
    # 2 = Micrometros (µm)
    # Se elige nm porque es la unidad estandar en optica/fotonica
    # y facilita clasificar UV (<380 nm), visible (380-780 nm) e IR (>780 nm)

    # --- FORMATO DE SALIDA ---
    "format": 2,
    # 0 = HTML (solo para ver en navegador, no procesable)
    # 1 = ASCII texto plano
    # 2 = CSV  <-- SELECCIONADO
    # 3 = Tab-delimited
    # CSV es el mas compatible con pandas y MySQL

    # --- QUE LINEAS INCLUIR ---
    "line_out": 0,
    # 0 = Todas las lineas (All)  <-- SELECCIONADO
    # 1 = Solo las que tienen probabilidades de transicion
    # 2 = Solo las que tienen clasificacion de nivel de energia
    # 3 = Solo las que tienen longitudes de onda observadas
    # Se elige 0 para no perder ninguna linea desde el inicio

    # --- UNIDADES DE ENERGIA ---
    "en_unit": 0,
    # 0 = cm-1 (numero de onda inverso)  <-- SELECCIONADO
    # 1 = eV
    # 2 = Rydberg
    # cm-1 es la unidad nativa del NIST para niveles atomicos
    # Se puede convertir a eV en la limpieza si se necesita

    # --- MOSTRAR RESULTADO COMPLETO ---
    "output": 0,
    # 0 = Todo en una sola pagina (in its entirety)  <-- SELECCIONADO
    # 1 = Paginado
    # Se elige 0 para obtener todos los datos de una sola vez

    # --- LONGITUD DE ONDA OBSERVADA ---
    "show_obs_wl": 1,
    # 1 = Incluir longitud de onda observada (medicion experimental real)
    # Es el dato mas valioso: medido directamente en laboratorio

    # --- LONGITUD DE ONDA RITZ ---
    "show_calc_wl": 1,
    # 1 = Incluir longitud de onda Ritz (calculada desde diferencia de niveles)
    # Complementa la observada: hay lineas sin medicion directa
    # pero con Ritz calculada

    # --- INCERTIDUMBRES ---
    "unc_out": 1,
    # 1 = Incluir incertidumbre de la longitud de onda
    # Util para evaluar la calidad/confiabilidad de cada dato

    # --- ORDEN DE SALIDA ---
    "order_out": 0,
    # 0 = Ordenado por longitud de onda  <-- SELECCIONADO
    # 1 = Ordenado por multiplete

    # --- CONVENCION DE LONGITUD DE ONDA (VACIO vs AIRE) ---
    "show_av": 2,
    # 2 = Vacio para < 2000 Å, aire para 2000-10000 Å, numero de onda para > 10000 Å
    # Esta es la convencion estandar del NIST

    # --- COEFICIENTE DE EINSTEIN Aki ---
    "A_out": 0,
    # 0 = Incluir Aki (probabilidad de transicion espontanea en s⁻¹)
    # Este es el parametro mas importante para fotonica:
    # indica que tan facilmente el atomo emite un foton

    # --- INTENSIDADES RELATIVAS ---
    "intens_out": "on",
    # Incluir intensidad relativa de cada linea
    # Indica el brillo relativo de cada linea espectral

    # --- TRANSICIONES PERMITIDAS (E1) ---
    "allowed_out": 1,
    # 1 = Incluir transiciones electricas de dipolo (E1)
    # Son las transiciones mas comunes y brillantes

    # --- TRANSICIONES PROHIBIDAS (M1, E2) ---
    "forbid_out": 1,
    # 1 = Incluir transiciones prohibidas (magneticas y cuadrupolo electrico)
    # Son mas raras pero existen en el dataset y dan info complementaria

    # --- CONFIGURACION ELECTRONICA ---
    "conf_out": "on",
    # Incluir la configuracion electronica de cada nivel
    # Ejemplo: 1s2 2s2 2p1
    # Necesaria para el modelo relacional (tabla Nivel_Energia)

    # --- TERMINO ESPECTROSCOPICO ---
    "term_out": "on",
    # Incluir el termino espectroscopico (ej. 2P*, 3D)
    # Describe el estado cuantico completo del nivel

    # --- ENERGIAS DE LOS NIVELES ---
    "enrg_out": "on",
    # Incluir la energia del nivel inferior y superior de cada transicion
    # Fundamental para calcular la energia del foton emitido

    # --- NUMERO CUANTICO J ---
    "J_out": "on",
    # Incluir el numero cuantico de momento angular total J
    # Necesario para calcular la degeneracion g = 2J+1

    # --- REFERENCIAS BIBLIOGRAFICAS ---
    "bibrefs": 1,
    # 1 = Incluir referencias de las probabilidades de transicion y de las lineas
    # Documenta la fuente cientifica de cada dato — requerido por la rubrica

    # --- PARAMETROS TECNICOS DEL FORMULARIO ---
    "submit": "Retrieve Data",
    "de": 0,
    "plot_out": 0,
    "I_scale_type": 1,
    "page_size": 15,
    "tsb_value": 0,
    "min_str": "",
    "max_str": "",
    "max_low_enrg": "",
    "max_upp_enrg": "",
    "min_accur": "",
    "min_intens": "",
}

print("Parametros de consulta definidos.")
print(f"Elementos a consultar: {ESPECTRA}")
print(f"Total de espectros: {len(ESPECTRA.split(';'))}")

---
## Celda 5 — Ejecutar la petición a la API del NIST

In [ ]:
print("Enviando peticion al NIST ASD...")
print(f"URL: {URL}")
print("Esto puede tardar entre 10 y 60 segundos dependiendo del volumen de datos.")
print("-" * 60)

response = requests.get(URL, params=parametros, timeout=120)

# Verificar que la peticion fue exitosa
print(f"Codigo de respuesta HTTP: {response.status_code}")

if response.status_code == 200:
    print("Peticion exitosa. Datos recibidos.")
    print(f"Tamano de la respuesta: {len(response.text):,} caracteres")
else:
    print(f"ERROR: La peticion fallo con codigo {response.status_code}")
    print(response.text[:500])

---
## Celda 6 — Verificar y previsualizar la respuesta

In [ ]:
# Mostrar las primeras lineas del texto recibido para verificar
# que el formato es CSV correcto

lineas_respuesta = response.text.split("\n")

print(f"Total de lineas en la respuesta: {len(lineas_respuesta):,}")
print("\n--- Primeras 10 lineas de la respuesta ---")
for i, linea in enumerate(lineas_respuesta[:10]):
    print(f"[{i}] {linea}")

---
## Celda 7 — Cargar los datos en un DataFrame de pandas

In [ ]:
# El NIST devuelve el CSV con algunas lineas de encabezado antes de los datos
# Se usa StringIO para que pandas pueda leer el texto directamente
# sin necesidad de guardarlo primero en disco

# Filtrar lineas que son comentarios del NIST (empiezan con # o son vacias)
lineas_datos = []
for linea in lineas_respuesta:
    # El NIST incluye lineas de separacion con guiones — las saltamos
    if linea.strip() and not linea.startswith("---"):
        lineas_datos.append(linea)

texto_limpio = "\n".join(lineas_datos)

# Leer el CSV con pandas
try:
    df_lines = pd.read_csv(
        StringIO(texto_limpio),
        sep=",",
        low_memory=False,    # Evita advertencias con columnas de tipo mixto
        on_bad_lines="warn"  # Avisa si hay filas con problemas sin detener la lectura
    )
    print(f"DataFrame cargado exitosamente.")
    print(f"Filas:    {df_lines.shape[0]:,}")
    print(f"Columnas: {df_lines.shape[1]}")
    print(f"\nNombres de columnas:")
    for col in df_lines.columns:
        print(f"  - {col}")
except Exception as e:
    print(f"Error al parsear el CSV: {e}")
    print("Revisa la celda anterior para ver el formato de la respuesta.")

---
## Celda 8 — Inspección inicial del DataFrame

In [ ]:
# Mostrar las primeras filas del DataFrame
print("=== Primeras 5 filas del dataset ===")
df_lines.head()

In [ ]:
# Tipos de datos de cada columna
print("=== Tipos de datos por columna ===")
print(df_lines.dtypes)

In [ ]:
# Resumen estadistico basico
print("=== Resumen estadistico ===")
df_lines.describe(include="all")

---
## Celda 9 — Guardar el archivo original sin modificaciones

In [ ]:
# Nombre del archivo original
NOMBRE_ARCHIVO = "datos_originales/nist_lines_H_Ne_original.csv"

# Guardar el texto crudo tal como vino del NIST (sin modificar)
# Esto cumple el requisito de conservar el archivo original intacto
with open(NOMBRE_ARCHIVO, "w", encoding="utf-8") as f:
    f.write(response.text)

print(f"Archivo original guardado en: {NOMBRE_ARCHIVO}")
print(f"Tamano del archivo: {os.path.getsize(NOMBRE_ARCHIVO):,} bytes")
print()
print("IMPORTANTE: Este archivo NO debe modificarse.")
print("Es la evidencia de la fuente de datos oficial (NIST ASD).")
print()
print("=" * 60)
print("DOCUMENTACION DE DESCARGA")
print("=" * 60)
print(f"Fuente:          NIST Atomic Spectra Database (ASD)")
print(f"URL base:        {URL}")
print(f"Tipo de dato:    Lineas espectrales")
print(f"Elementos:       H, He, Li, Be, B, C, N, O, F, Ne (neutros e ionizados)")
print(f"Rango lambda:    Completo (sin restriccion)")
print(f"Unidades lambda: Nanometros (nm)")
print(f"Unidades E:      cm-1")
print(f"Formato salida:  CSV")
print(f"Fecha descarga:  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Filas obtenidas: {df_lines.shape[0]:,}")
print(f"Columnas:        {df_lines.shape[1]}")
print(f"Archivo:         {NOMBRE_ARCHIVO}")

---
## Parte A completada

Archivo: `datos_originales/nist_lines_H_Ne_original.csv`. **No modificarlo.**

Sigue la Parte B (niveles). No hace falta un notebook aparte.


---
# Parte B — Niveles de energía (`energy1.pl`)

El formulario de niveles acepta **un espectro a la vez**, por eso hay **18 peticiones**. Cada fila es un nivel de energía. Al final se guardan un CSV por espectro y el consolidado.


---
## Celda 4 — Definición de la URL y lista de espectros a descargar

In [ ]:
# URL del endpoint de la API del NIST ASD para niveles de energia
# Es diferente al de lineas espectrales (energy1.pl vs lines1.pl)
URL = "http://physics.nist.gov/cgi-bin/ASD/energy1.pl"

# Lista de todos los espectros a descargar
# Formato: 'SIMBOLO ESTADO' donde estado I = neutro, II = primer ionizado
# El formulario de niveles solo acepta UN espectro por peticion
# por eso se hace una peticion separada para cada uno
ESPECTRA = [
    "H I",    # Hidrogeno neutro
    "He I",   # Helio neutro
    "Li I",   # Litio neutro
    "Li II",  # Litio primer ionizado
    "Be I",   # Berilio neutro
    "Be II",  # Berilio primer ionizado
    "B I",    # Boro neutro
    "B II",   # Boro primer ionizado
    "C I",    # Carbono neutro
    "C II",   # Carbono primer ionizado
    "N I",    # Nitrogeno neutro
    "N II",   # Nitrogeno primer ionizado
    "O I",    # Oxigeno neutro
    "O II",   # Oxigeno primer ionizado
    "F I",    # Fluor neutro
    "F II",   # Fluor primer ionizado
    "Ne I",   # Neon neutro
    "Ne II",  # Neon primer ionizado
]

print(f"Total de espectros a descargar: {len(ESPECTRA)}")
print("Lista de espectros:")
for esp in ESPECTRA:
    print(f"  - {esp}")

---
## Celda 5 — Definición de parámetros base de la consulta

Cada parámetro corresponde exactamente a una opción del formulario web en:  
https://physics.nist.gov/PhysRefData/ASD/levels_form.html

In [ ]:
# Parametros base que se usaran para TODOS los espectros
# Solo cambiara el campo 'spectrum' en cada peticion

PARAMETROS_BASE = {

    # --- ESPECTRO (se reemplazara en cada iteracion) ---
    # "spectrum": se asigna dinamicamente en el loop

    # --- UNIDADES DE ENERGIA ---
    "units": 1,
    # 0 = cm-1 (numero de onda)
    # 1 = eV (electronvoltios)  <-- SELECCIONADO
    # 2 = Rydberg
    # 3 = Hartree
    # 4 = GHz
    # Se elige eV porque es la unidad mas universal en fisica moderna
    # y la mas intuitiva para comparar entre elementos

    # --- FORMATO DE SALIDA ---
    "format": 2,
    # 0 = HTML
    # 1 = ASCII
    # 2 = CSV  <-- SELECCIONADO
    # 3 = Tab-delimited

    # --- MOSTRAR RESULTADO COMPLETO ---
    "output": 0,
    # 0 = Todo en una sola pagina (in its entirety)  <-- SELECCIONADO
    # 1 = Paginado

    # --- ORDENAMIENTO ---
    "order": 0,
    # 0 = Ordenado por energia (Energy ordered)  <-- SELECCIONADO
    # 1 = Ordenado por termino (Term ordered)

    # --- INFORMACION DE NIVEL: CONFIGURACION PRINCIPAL ---
    "show_conf": 1,
    # 1 = Incluir la configuracion electronica principal
    # Ejemplo: 1s2 2s2 2p1
    # Necesaria para identificar cada nivel en el modelo relacional

    # --- INFORMACION DE NIVEL: TERMINO ESPECTROSCOPICO ---
    "show_term": 1,
    # 1 = Incluir el termino espectroscopico (ej. 2P*, 3D, 1S)
    # Describe el estado cuantico completo del nivel atomico

    # --- INFORMACION DE NIVEL: INCERTIDUMBRE ---
    "show_level_unc": 1,
    # 1 = Incluir la incertidumbre de la energia del nivel
    # Indica la calidad y precision del dato experimental

    # --- INFORMACION DE NIVEL: NUMERO CUANTICO J ---
    "show_j": 1,
    # 1 = Incluir el numero cuantico de momento angular total J
    # Necesario para calcular la degeneracion: g = 2J + 1

    # --- INFORMACION DE NIVEL: DEGENERACION g ---
    "show_g": 1,
    # 1 = Incluir la degeneracion estadistica g del nivel
    # g = 2J + 1, aparece en el calculo de la fuerza de oscilador

    # --- INFORMACION DE NIVEL: IDs DE NIVEL ---
    "show_level_id": 1,
    # 1 = Incluir los identificadores internos del nivel en la BD del NIST
    # Util para hacer JOIN con la tabla de lineas espectrales

    # --- FACTOR DE LANDE-g ---
    "show_lande_g": 1,
    # 1 = Incluir el factor de Lande-g del nivel
    # Describe como responde el nivel a campos magneticos externos
    # Relevante para efectos Zeeman y aplicaciones en fotonica cuantica

    # --- PORCENTAJES DE COMPOSICION (Leading percentages) ---
    "show_perc": 1,
    # 1 = Incluir los porcentajes de composicion de la funcion de onda
    # Indica que fraccion del nivel corresponde a cada configuracion
    # Util para caracterizar niveles mixtos en atomos multielectronicos

    # --- REFERENCIAS BIBLIOGRAFICAS ---
    "biblio": 1,
    # 1 = Incluir referencias de la fuente de cada nivel de energia
    # Documenta la procedencia cientifica — requerido por la rubrica

    # --- PARAMETROS TECNICOS ---
    "submit": "Retrieve Data",
    "page_size": 15,
    "temp": "",    # Temperatura para funcion de particion (no requerida)
}

print("Parametros base definidos correctamente.")
print(f"Unidades de energia: eV")
print(f"Formato de salida: CSV")
print(f"Columnas activadas: configuracion, termino, incertidumbre, J, g, Level IDs, Lande-g, leading percentages, referencias")

---
## Celda 6 — Función auxiliar para limpiar y parsear la respuesta del NIST

In [ ]:
def parsear_respuesta_nist(texto_respuesta, nombre_espectro):
    """
    Convierte el texto CSV del NIST en un DataFrame de pandas.
    Agrega una columna 'espectro' para identificar el elemento
    despues de consolidar todos los archivos en uno solo.
    
    Parametros:
        texto_respuesta (str): Texto crudo devuelto por la API del NIST
        nombre_espectro (str): Nombre del espectro consultado (ej. 'H I')
    
    Retorna:
        pd.DataFrame o None si hubo error
    """
    try:
        # Filtrar lineas vacias y separadores del NIST
        lineas = texto_respuesta.split("\n")
        lineas_utiles = [
            linea for linea in lineas
            if linea.strip() and not linea.startswith("---")
        ]
        texto_limpio = "\n".join(lineas_utiles)
        
        # Parsear el CSV
        df = pd.read_csv(
            StringIO(texto_limpio),
            sep=",",
            low_memory=False,
            on_bad_lines="warn"
        )
        
        # Agregar columna identificadora del espectro
        # Esto es fundamental para poder distinguir los niveles
        # cuando se consoliden todos los archivos en uno solo
        df.insert(0, "espectro", nombre_espectro)
        
        return df
    
    except Exception as e:
        print(f"  ERROR al parsear {nombre_espectro}: {e}")
        return None

print("Funcion auxiliar definida.")

---
## Celda 7 — Descarga automática de todos los espectros

Esta celda hace 18 peticiones al NIST, una por cada espectro.  
Guarda cada resultado como archivo individual y al final los consolida en uno solo.

In [ ]:
# Lista para acumular todos los DataFrames descargados
lista_dfs = []

# Registro de resultados de cada descarga
registro = []

print("Iniciando descarga de niveles de energia...")
print(f"Total de espectros: {len(ESPECTRA)}")
print("=" * 60)

for i, espectro in enumerate(ESPECTRA, start=1):
    
    print(f"\n[{i:02d}/{len(ESPECTRA)}] Descargando: {espectro} ...")
    
    # Construir parametros para esta peticion especifica
    # Copiamos los parametros base y agregamos el espectro
    parametros = PARAMETROS_BASE.copy()
    parametros["spectrum"] = espectro
    
    try:
        # Hacer la peticion HTTP al NIST
        response = requests.get(URL, params=parametros, timeout=60)
        
        # Verificar codigo de respuesta
        if response.status_code != 200:
            print(f"  ERROR HTTP {response.status_code} para {espectro}")
            registro.append({"espectro": espectro, "estado": "ERROR HTTP", "filas": 0})
            continue
        
        # Guardar el texto crudo del NIST como archivo original individual
        # Nombre de archivo: reemplazar espacios por guion bajo
        nombre_archivo = espectro.replace(" ", "").replace("I", "I").lower()
        nombre_archivo = f"datos_originales/niveles/nist_levels_{espectro.replace(' ', '_')}_original.csv"
        
        with open(nombre_archivo, "w", encoding="utf-8") as f:
            f.write(response.text)
        
        # Parsear la respuesta a DataFrame
        df = parsear_respuesta_nist(response.text, espectro)
        
        if df is not None and len(df) > 0:
            lista_dfs.append(df)
            print(f"  OK — {len(df):,} niveles descargados — guardado en {nombre_archivo}")
            registro.append({"espectro": espectro, "estado": "OK", "filas": len(df)})
        else:
            print(f"  AVISO — Respuesta vacia o sin datos para {espectro}")
            registro.append({"espectro": espectro, "estado": "VACIO", "filas": 0})
    
    except requests.exceptions.Timeout:
        print(f"  ERROR — Tiempo de espera agotado para {espectro}")
        registro.append({"espectro": espectro, "estado": "TIMEOUT", "filas": 0})
    
    except Exception as e:
        print(f"  ERROR inesperado para {espectro}: {e}")
        registro.append({"espectro": espectro, "estado": f"ERROR: {e}", "filas": 0})
    
    # Pausa de 1 segundo entre peticiones
    # Buena practica para no saturar el servidor del NIST
    time.sleep(1)

print("\n" + "=" * 60)
print("DESCARGA COMPLETADA")
print("=" * 60)

# Mostrar resumen de descargas
df_registro = pd.DataFrame(registro)
print(df_registro.to_string(index=False))
print(f"\nTotal espectros exitosos: {df_registro[df_registro['estado']=='OK'].shape[0]}/{len(ESPECTRA)}")

---
## Celda 8 — Consolidar todos los espectros en un solo DataFrame

In [ ]:
# Unir todos los DataFrames individuales en uno solo
# ignore_index=True renumera los indices de 0 a N

if lista_dfs:
    df_levels = pd.concat(lista_dfs, ignore_index=True)
    
    print("DataFrame consolidado creado.")
    print(f"Total de filas:    {df_levels.shape[0]:,}")
    print(f"Total de columnas: {df_levels.shape[1]}")
    print(f"\nNombres de columnas:")
    for col in df_levels.columns:
        print(f"  - {col}")
else:
    print("ERROR: No se pudo descargar ningun espectro. Revisar conexion a internet.")

---
## Celda 9 — Inspección inicial del DataFrame consolidado

In [ ]:
# Primeras filas del DataFrame consolidado
print("=== Primeras 5 filas del dataset consolidado ===")
df_levels.head()

In [ ]:
# Cuantos niveles hay por espectro
print("=== Niveles de energia por espectro ===")
conteo = df_levels.groupby("espectro").size().reset_index(name="num_niveles")
print(conteo.to_string(index=False))

In [ ]:
# Tipos de datos
print("=== Tipos de datos por columna ===")
print(df_levels.dtypes)

In [ ]:
# Resumen estadistico
print("=== Resumen estadistico ===")
df_levels.describe(include="all")

---
## Celda 10 — Guardar el archivo consolidado original

In [ ]:
# Nombre del archivo consolidado
NOMBRE_CONSOLIDADO = "datos_originales/nist_levels_todos_original.csv"

# Guardar el DataFrame consolidado como CSV
# index=False evita guardar el numero de fila como columna extra
df_levels.to_csv(NOMBRE_CONSOLIDADO, index=False, encoding="utf-8")

print(f"Archivo consolidado guardado en: {NOMBRE_CONSOLIDADO}")
print(f"Tamano del archivo: {os.path.getsize(NOMBRE_CONSOLIDADO):,} bytes")
print()
print("IMPORTANTE: Este archivo NO debe modificarse.")
print("Es la evidencia de la fuente de datos oficial (NIST ASD).")
print()
print("=" * 60)
print("DOCUMENTACION DE DESCARGA")
print("=" * 60)
print(f"Fuente:            NIST Atomic Spectra Database (ASD)")
print(f"URL base:          {URL}")
print(f"Tipo de dato:      Niveles de energia atomicos")
print(f"Elementos:         H, He, Li, Be, B, C, N, O, F, Ne (neutros e ionizados)")
print(f"Unidades energia:  eV (electronvoltios)")
print(f"Formato salida:    CSV")
print(f"Fecha descarga:    {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Espectros OK:      {df_registro[df_registro['estado']=='OK'].shape[0]}/{len(ESPECTRA)}")
print(f"Total filas:       {df_levels.shape[0]:,}")
print(f"Total columnas:    {df_levels.shape[1]}")
print(f"Archivo:           {NOMBRE_CONSOLIDADO}")
print(f"Archivos individuales en: datos_originales/niveles/")

---
## Celda 11 — Verificar la estructura de archivos generados

In [ ]:
# Listar todos los archivos en la carpeta datos_originales
print("=== Archivos en datos_originales/ ===")

for raiz, carpetas, archivos in os.walk("datos_originales"):
    nivel = raiz.replace("datos_originales", "").count(os.sep)
    sangria = "  " * nivel
    print(f"{sangria}{os.path.basename(raiz)}/")
    sub_sangria = "  " * (nivel + 1)
    for archivo in sorted(archivos):
        ruta = os.path.join(raiz, archivo)
        tamano = os.path.getsize(ruta)
        print(f"{sub_sangria}{archivo}  ({tamano:,} bytes)")

---
## Descarga completada (partes A y B)

| Archivo | Contenido |
|---|---|
| `datos_originales/nist_lines_H_Ne_original.csv` | Líneas espectrales (Parte A) |
| `datos_originales/nist_levels_todos_original.csv` | Niveles consolidados (Parte B) |
| `datos_originales/niveles/nist_levels_*_original.csv` | Un archivo por espectro |

**No modificar ninguno.** Siguiente paso: `02_exploracion_caracterizacion_datos.ipynb`.
